In [1]:
import pandas as pandas
import numpy as np 
import matplotlib.pyplot as plt

In [2]:
X_train = np.array([[1, 1, 1],
[0, 0, 1],
 [0, 1, 0],
 [1, 0, 1],
 [1, 1, 1],
 [1, 1, 0],
 [0, 0, 0],
 [1, 1, 0],
 [0, 1, 0],
 [0, 1, 0]])

y_train = np.array([1, 1, 0, 0, 1, 1, 0, 1, 0, 0])

In [3]:
#for instance, the first example
X_train[0]

array([1, 1, 1])

In [4]:
def entropy(p):
    if p==0 or p==1:
        return 0
    else:
        return -p*np.log2(p) - (1-p) * np.log2(1-p)
print(entropy(0.5)) 
    
    

1.0


In [5]:
def split_indices(X, index_feature):
    """Given a dataset and a index feature, return two lists for the two split nodes, the left node has the animals that have 
    that feature = 1 and the right node those that have the feature = 0 
    index feature = 0 => ear shape
    index feature = 1 => face shape
    index feature = 2 => whiskers
    """
    left_indices = []
    right_indices = []
    for i,x in enumerate(X):
        if x[index_feature] == 1:
            left_indices.append(i)
        else:
            right_indices.append(i)
    return left_indices, right_indices

In [6]:
split_indices(X_train, 0)

([0, 3, 4, 5, 7], [1, 2, 6, 8, 9])

In [7]:
def weighted_entropy(X,y,left_indices,right_indices):
    """
    This function takes the splitted dataset, the indices we chose to split and returns the weighted entropy.
    """
    w_left = len(left_indices)/len(X)
    w_right = len(right_indices)/len(X)
    p_left = sum(y[left_indices])/len(left_indices)
    p_right = sum(y[right_indices])/len(right_indices)
    
    weighted_entropy = w_left * entropy(p_left) + w_right * entropy(p_right)
    return weighted_entropy

In [8]:
left_indices, right_indices = split_indices(X_train, 0)
weighted_entropy(X_train, y_train, left_indices, right_indices)

np.float64(0.7219280948873623)

In [9]:
def information_gain(X, y, left_indices, right_indices):
    """
    Here, X has the elements in the node and y is theirs respectives classes
    """
    p_node = sum(y)/len(y)
    h_node = entropy(p_node)
    w_entropy = weighted_entropy(X,y,left_indices,right_indices)
    return h_node - w_entropy

In [10]:
information_gain(X_train, y_train, left_indices, right_indices)

np.float64(0.2780719051126377)

In [11]:
for i, feature_name in enumerate(['Ear Shape', 'Face Shape', 'Whiskers']):
    left_indices, right_indices = split_indices(X_train, i)
    i_gain = information_gain(X_train, y_train, left_indices, right_indices)
    print(f"Feature: {feature_name}, information gain if we split the root node using this feature: {i_gain:.2f}")
    

Feature: Ear Shape, information gain if we split the root node using this feature: 0.28
Feature: Face Shape, information gain if we split the root node using this feature: 0.03
Feature: Whiskers, information gain if we split the root node using this feature: 0.12


In [12]:
def build_tree_recursive(X, y, indices, node_name, max_depth, current_depth, tree):
	"""
	Recursively build a decision tree.
	"""
	# Base cases
	if current_depth == max_depth or len(set(y[indices])) == 1:
		# Leaf node: determine majority class
		classes, counts = np.unique(y[indices], return_counts=True)
		majority_class = classes[np.argmax(counts)]
		tree.append({'name': node_name, 'type': 'leaf', 'class': majority_class, 'indices': indices})
		return
	
	# Find the best feature to split on
	best_feature = None
	best_gain = -1
	for i in range(X.shape[1]):
		left_indices, right_indices = split_indices(X[indices], i)
		# Adjust indices to original
		left_orig = [indices[j] for j in left_indices]
		right_orig = [indices[j] for j in right_indices]
		if len(left_orig) == 0 or len(right_orig) == 0:
			continue
		gain = information_gain(X[indices], y[indices], left_indices, right_indices)
		if gain > best_gain:
			best_gain = gain
			best_feature = i
	
	if best_feature is None:
		# No split possible, make leaf
		classes, counts = np.unique(y[indices], return_counts=True)
		majority_class = classes[np.argmax(counts)]
		tree.append({'name': node_name, 'type': 'leaf', 'class': majority_class, 'indices': indices})
		return
	
	# Split
	left_indices, right_indices = split_indices(X[indices], best_feature)
	left_orig = [indices[j] for j in left_indices]
	right_orig = [indices[j] for j in right_indices]
	
	# Add node to tree
	tree.append({'name': node_name, 'type': 'node', 'feature': best_feature, 'left': len(tree)+1, 'right': len(tree)+2, 'indices': indices})
	
	# Recurse
	build_tree_recursive(X, y, left_orig, f"{node_name}_left", max_depth, current_depth+1, tree)
	build_tree_recursive(X, y, right_orig, f"{node_name}_right", max_depth, current_depth+1, tree)

def generate_tree_viz(indices, y, tree):
	"""
	Simple visualization: print the tree structure.
	"""
	def print_tree(node_index, indent=0):
		node = tree[node_index]
		print('  ' * indent + node['name'])
		if node['type'] == 'node':
			print('  ' * (indent+1) + f"Feature {node['feature']}")
			print_tree(node['left'], indent+2)
			print_tree(node['right'], indent+2)
		else:
			print('  ' * (indent+1) + f"Class: {node['class']}")
	
	print_tree(0)

tree = []
build_tree_recursive(X_train, y_train, [0,1,2,3,4,5,6,7,8,9], "Root", max_depth=1, current_depth=0, tree=tree)
generate_tree_viz([0,1,2,3,4,5,6,7,8,9], y_train, tree)

Root
  Feature 0
    Root_left
      Class: 1
    Root_right
      Class: 0


In [13]:
tree = []
build_tree_recursive(X_train, y_train, [0,1,2,3,4,5,6,7,8,9], "Root", max_depth=2, current_depth=0, tree = tree)
generate_tree_viz([0,1,2,3,4,5,6,7,8,9], y_train, tree)

Root
  Feature 0
    Root_left
      Feature 1
        Root_left_left
          Class: 1
        Root_left_right
          Class: 0
    Root_left_left
      Class: 1
